# 01. データ探索 (Data Exploration)

このノートブックでは、競馬予測MLシステムのデータ概要を確認します。

## 目次
1. セットアップ
2. データの読み込み
3. 基本統計量の確認
4. 欠損値の分析
5. データ型の確認
6. レコード数の推移

## 1. セットアップ

In [ ]:
# 必要なライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings('ignore')

# 日本語フォント設定（macOS）
plt.rcParams['font.family'] = 'Hiragino Sans'
plt.rcParams['axes.unicode_minus'] = False

# プロット設定
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# 表示設定
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Setup complete!')

In [ ]:
# 環境変数の読み込み
load_dotenv()

PROJECT_ID = os.environ.get('GCP_PROJECT_ID')
print(f'Project ID: {PROJECT_ID}')

# BigQueryクライアントの初期化
client = bigquery.Client(project=PROJECT_ID)

## 2. データの読み込み

BigQueryから各テーブルのデータを確認します。

In [ ]:
# テーブル一覧を取得
def list_tables(dataset_id):
    """データセット内のテーブル一覧を取得"""
    tables = client.list_tables(f'{PROJECT_ID}.{dataset_id}')
    return [table.table_id for table in tables]

# rawデータセットのテーブル一覧
print('=== raw データセット ===')
try:
    raw_tables = list_tables('raw')
    for table in raw_tables:
        print(f'  - {table}')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# 各テーブルのレコード数を確認
def get_table_info(dataset_id, table_id):
    """テーブルの基本情報を取得"""
    query = f"""
    SELECT 
        COUNT(*) as row_count,
        MIN(race_date) as min_date,
        MAX(race_date) as max_date
    FROM `{PROJECT_ID}.{dataset_id}.{table_id}`
    """
    try:
        result = client.query(query).to_dataframe()
        return result.iloc[0].to_dict()
    except Exception as e:
        return {'error': str(e)}

# 主要テーブルの情報を取得
tables_to_check = [
    ('raw', 'race_info'),
    ('raw', 'horse_results'),
]

print('=== テーブル基本情報 ===')
for dataset_id, table_id in tables_to_check:
    info = get_table_info(dataset_id, table_id)
    print(f'\n{dataset_id}.{table_id}:')
    for key, value in info.items():
        print(f'  {key}: {value}')

## 3. 基本統計量の確認

In [ ]:
# race_infoテーブルのサンプルデータを取得
query_race_info = f"""
SELECT *
FROM `{PROJECT_ID}.raw.race_info`
LIMIT 1000
"""

df_race_info = client.query(query_race_info).to_dataframe()
print(f'race_info サンプル数: {len(df_race_info)}')
print(f'カラム数: {len(df_race_info.columns)}')
print(f'\nカラム一覧:')
print(df_race_info.columns.tolist())

In [ ]:
# race_infoの基本統計量
print('=== race_info 基本統計量 ===')
df_race_info.describe()

In [ ]:
# horse_resultsテーブルのサンプルデータを取得
query_horse_results = f"""
SELECT *
FROM `{PROJECT_ID}.raw.horse_results`
LIMIT 1000
"""

df_horse_results = client.query(query_horse_results).to_dataframe()
print(f'horse_results サンプル数: {len(df_horse_results)}')
print(f'カラム数: {len(df_horse_results.columns)}')
print(f'\nカラム一覧:')
print(df_horse_results.columns.tolist())

In [ ]:
# horse_resultsの基本統計量
print('=== horse_results 基本統計量 ===')
df_horse_results.describe()

## 4. 欠損値の分析

In [ ]:
def analyze_missing_values(df, table_name):
    """欠損値の分析"""
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    
    missing_df = pd.DataFrame({
        'カラム名': missing.index,
        '欠損数': missing.values,
        '欠損率(%)': missing_pct.values
    })
    missing_df = missing_df[missing_df['欠損数'] > 0].sort_values('欠損率(%)', ascending=False)
    
    print(f'\n=== {table_name} 欠損値分析 ===')
    print(f'総カラム数: {len(df.columns)}')
    print(f'欠損があるカラム数: {len(missing_df)}')
    
    if len(missing_df) > 0:
        print(f'\n欠損率が高いカラム（上位10件）:')
        display(missing_df.head(10))
        
        # 欠損率の可視化
        if len(missing_df) > 0:
            plt.figure(figsize=(12, 6))
            top_missing = missing_df.head(20)
            plt.barh(top_missing['カラム名'], top_missing['欠損率(%)'])
            plt.xlabel('欠損率 (%)')
            plt.title(f'{table_name} - 欠損率（上位20カラム）')
            plt.tight_layout()
            plt.show()
    else:
        print('欠損値はありません')
    
    return missing_df

In [ ]:
# race_infoの欠損値分析
missing_race_info = analyze_missing_values(df_race_info, 'race_info')

In [ ]:
# horse_resultsの欠損値分析
missing_horse_results = analyze_missing_values(df_horse_results, 'horse_results')

## 5. データ型の確認

In [ ]:
def analyze_data_types(df, table_name):
    """データ型の分析"""
    dtype_df = pd.DataFrame({
        'カラム名': df.columns,
        'データ型': df.dtypes.values,
        'ユニーク数': [df[col].nunique() for col in df.columns],
        'サンプル値': [df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None for col in df.columns]
    })
    
    print(f'\n=== {table_name} データ型分析 ===')
    print(f'\nデータ型の分布:')
    print(df.dtypes.value_counts())
    
    return dtype_df

In [ ]:
# race_infoのデータ型分析
dtype_race_info = analyze_data_types(df_race_info, 'race_info')
dtype_race_info.head(20)

In [ ]:
# horse_resultsのデータ型分析
dtype_horse_results = analyze_data_types(df_horse_results, 'horse_results')
dtype_horse_results.head(20)

## 6. レコード数の推移

In [ ]:
# 月別レコード数の推移を確認
query_monthly_counts = f"""
SELECT 
    FORMAT_DATE('%Y-%m', race_date) as year_month,
    COUNT(DISTINCT race_id) as race_count,
    COUNT(*) as total_records
FROM `{PROJECT_ID}.raw.race_info`
GROUP BY year_month
ORDER BY year_month
"""

try:
    df_monthly = client.query(query_monthly_counts).to_dataframe()
    
    # 可視化
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # レース数の推移
    axes[0].plot(df_monthly['year_month'], df_monthly['race_count'], marker='o', markersize=3)
    axes[0].set_title('月別レース数の推移')
    axes[0].set_xlabel('年月')
    axes[0].set_ylabel('レース数')
    axes[0].tick_params(axis='x', rotation=45)
    # X軸のラベルを間引く
    n = len(df_monthly)
    step = max(1, n // 20)
    axes[0].set_xticks(range(0, n, step))
    axes[0].set_xticklabels(df_monthly['year_month'].iloc[::step])
    
    # レコード数の推移
    axes[1].plot(df_monthly['year_month'], df_monthly['total_records'], marker='o', markersize=3, color='orange')
    axes[1].set_title('月別レコード数の推移')
    axes[1].set_xlabel('年月')
    axes[1].set_ylabel('レコード数')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].set_xticks(range(0, n, step))
    axes[1].set_xticklabels(df_monthly['year_month'].iloc[::step])
    
    plt.tight_layout()
    plt.show()
    
    print('\n=== 月別統計サマリー ===')
    print(f'期間: {df_monthly["year_month"].min()} ～ {df_monthly["year_month"].max()}')
    print(f'総月数: {len(df_monthly)}')
    print(f'月平均レース数: {df_monthly["race_count"].mean():.1f}')
    print(f'月平均レコード数: {df_monthly["total_records"].mean():.1f}')
except Exception as e:
    print(f'Error: {e}')

## まとめ

このノートブックでは以下の分析を行いました：

1. **テーブル構造の確認**: BigQueryに格納されているテーブル一覧とカラム情報
2. **基本統計量**: 数値カラムの分布（平均、標準偏差、最小/最大値など）
3. **欠損値分析**: 各カラムの欠損率と欠損パターン
4. **データ型分析**: 各カラムのデータ型とユニーク数
5. **時系列トレンド**: 月別のレコード数推移

### 次のステップ
- `02_race_analysis.ipynb`: レース条件と結果の関係分析
- `03_horse_analysis.ipynb`: 馬の成績分析
- `04_feature_correlation.ipynb`: 特徴量の相関分析